## 📓 Notebook 03: Train/Test Split

**Skills practiced:** `train_test_split` with and without `stratify=`, `.fit_transform()` on train vs `.transform()` on test, spotting data leakage.

**Goal:** Split the complaints correctly, fit the vectorizer on train only, and demonstrate the "wrong way" (fitting on test) to see the leakage problem first-hand.

In [113]:
import pandas as pd


from scipy.sparse import save_npz

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer


In [114]:
df = pd.read_csv('../data/processed/complaints_sample.csv')

In [115]:
X_train, X_test, y_train, y_test = train_test_split(
   df['Issue'], df['Product'], test_size=0.2,
   random_state=42)

In [116]:
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)

Product
Checking or savings account    0.1034
Credit card or prepaid card    0.1030
Credit reporting               0.1021
Bank account or service        0.1009
Credi and other personal       0.1008
Student loan                   0.0995
Credit card                    0.0986
Consumer Loan                  0.0985
Debt collection                0.0977
Mortgage                       0.0955
Name: proportion, dtype: float64

In [117]:
X_train, X_test, y_train, y_test = train_test_split(
   df['Issue'], df['Product'], test_size=0.2,
   random_state=42, stratify=df['Product'])

In [118]:
pd.DataFrame({'train': y_train.value_counts(normalize=True),
              'test': y_test.value_counts(normalize=True)})

,train,test
Product,,
Bank account or service,0.1,0.1
Checking or savings account,0.1,0.1
Consumer Loan,0.1,0.1
Credi and other personal,0.1,0.1
Credit card,0.1,0.1
Credit card or prepaid card,0.1,0.1
Credit reporting,0.1,0.1
Debt collection,0.1,0.1
Mortgage,0.1,0.1


In [119]:
vec = TfidfVectorizer(stop_words='english')
train_matrix = vec.fit_transform(X_train)
#converts the test texts using what vec already learned.
test_matrix = vec.transform(X_test)

correct_size = len(vec.vocabulary_)

In [120]:
save_npz('../data/processed/x_train.npz', train_matrix)
save_npz('../data/processed/x_test.npz', test_matrix)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
X_test.to_csv('../data/processed/x_text.csv', index=False)

In [121]:
wrong_vec = TfidfVectorizer(stop_words='english')
#As a deliberate demonstration
wrong_matrix = wrong_vec.fit_transform(X_test)

wrong_size = len(wrong_vec.vocabulary_)

In [122]:
print(correct_size, wrong_size)
print(train_matrix.shape[1], test_matrix.shape[1], wrong_matrix.shape[1])

166 160
166 166 160


Fitting the vectorizer on `X_test` lets the test data shape the vocabulary and IDF weights, so the model's evaluation is influenced by information it should never have seen, which makes the score look better than it would on truly new data.